In [1]:
# ==========================================
# MODEL 1 - MARGIN COMPRESSION MODEL
# ==========================================

import numpy as np
import pandas as pd


In [2]:
df_merged_prediction = pd.read_csv(r"C:\Users\kiano.keynejad\Documents\1st Oct May\df_merged_May.csv")

In [3]:
# -----------------------------------------------------
# Proposed Pricing by Rate Segment
# -----------------------------------------------------

df_merged_prediction['Proposed MSF CC %'] = np.select(
    [
        df_merged_prediction['Tiers'] == '0-50K',
        df_merged_prediction['Tiers'] == '50K-100K',
        df_merged_prediction['Tiers'] == '100K+'
    ],
    [
        0.90,
        0.80,
        0.70
    ]
)

df_merged_prediction['Proposed MSF DC $'] = np.select(
    [
        df_merged_prediction['Tiers'] == '0-50K',
        df_merged_prediction['Tiers'] == '50K-100K',
        df_merged_prediction['Tiers'] == '100K+'
    ],
    [
        0.30,
        0.25,
        0.20
    ]
)

In [4]:
# -----------------------------------------------------
# Split TTV into Tap and EFTPOS Non-Tap
# -----------------------------------------------------

df_merged_prediction['Tap TTV'] = (
    df_merged_prediction['Value of Transactions']
    * (df_merged_prediction['% Total Tap'] / 100)
)

df_merged_prediction['EFTPOS Non-Tap TTV'] = (
    df_merged_prediction['Value of Transactions']
    * (df_merged_prediction['% Eftpos (non-Tap)'] / 100)
)

In [5]:
# -----------------------------------------------------
# Future Tap Revenue
# -----------------------------------------------------

df_merged_prediction['Future Tap Revenue'] = (
    df_merged_prediction['Tap TTV']
    * (df_merged_prediction['Proposed MSF CC %'] / 100)
)

In [6]:
# -----------------------------------------------------
# Future EFTPOS Non-Tap Revenue
# -----------------------------------------------------
# Proxy calculation using EFTPOS Non-Tap TTV
# because transaction count is unavailable

df_merged_prediction['Future EFTPOS Revenue'] = (
    df_merged_prediction['EFTPOS Non-Tap TTV']
    * (df_merged_prediction['Proposed MSF DC $'] / 100)
)

In [7]:
# -----------------------------------------------------
# Total Future Revenue ( Future MSF )
# -----------------------------------------------------

df_merged_prediction['Future Revenue'] = (
    df_merged_prediction['Future Tap Revenue']
    + df_merged_prediction['Future EFTPOS Revenue']
)

In [8]:
df_merged_prediction.head()

,Merchant ID,Company Name,MCC code,Business Industry,Price Model,MSF CC %,MSF DC $,State,PostCode,Default Brand,...,% Credit,% International,% Total Tap,Proposed MSF CC %,Proposed MSF DC $,Tap TTV,EFTPOS Non-Tap TTV,Future Tap Revenue,Future EFTPOS Revenue,Future Revenue
0,SP7030000005012,Anatolia Gozleme Kitchen,5812,"Eating Places, Restaurants",SmartCharge,1.20,0.30,VIC,3181.0,Smartpay,...,19.4,2.1,99.75,0.7,0.2,134290.282875,201.940275,940.031980,0.403881,940.435861
1,SP7040000004162,Aussizz Migration And Education acc 2,8111,"Attorneys, Legal Services",SmartCharge,1.65,0.30,QLD,4000.0,Smartpay,...,0.0,0.0,100.00,0.9,0.3,6719.130000,0.000000,60.472170,0.000000,60.472170
2,SP7030000005066,Bella and Home Decor,5712,"Furniture, Home Furnishings, and Equipment Sto...",Simple rate,1.05,0.24,VIC,3803.0,Smartpay,...,23.0,0.0,96.10,0.9,0.3,22937.051900,930.848100,206.433467,2.792544,209.226011
3,SP7030000004158,Afghan Kebab Campbellfield,5812,"Eating Places, Restaurants",SmartCharge,1.10,0.25,VIC,3061.0,Smartpay,...,7.9,0.6,99.55,0.7,0.2,120790.007910,546.012090,845.530055,1.092024,846.622080
4,SP7030000002212,Byblos Market,5499,Misc. Food Stores - Convenience & Specialty Ma...,SmartCharge,1.65,0.30,VIC,3023.0,Smartpay,...,2.0,0.0,100.00,0.9,0.3,7335.990000,0.000000,66.023910,0.000000,66.023910


In [9]:
# -----------------------------------------------------
# Future Margin
# -----------------------------------------------------

df_merged_prediction['Future Margin'] = (
    df_merged_prediction['Future Revenue']
    - df_merged_prediction['COGS']
)

In [10]:
# -----------------------------------------------------
# Margin Compression $
# -----------------------------------------------------

df_merged_prediction['Margin Compression $'] = (
    df_merged_prediction['Future Margin']
    - df_merged_prediction['MSF Margin']
)

In [11]:
# -----------------------------------------------------
# Margin Compression %
# -----------------------------------------------------

df_merged_prediction['Margin Compression %'] = np.where(
    df_merged_prediction['MSF Margin'] != 0,
    (
        df_merged_prediction['Margin Compression $']
        / df_merged_prediction['MSF Margin']
    ) * 100,
    np.nan
)

In [12]:
# -----------------------------------------------------
# Risk Classification
# -----------------------------------------------------

conditions = [
    df_merged_prediction['Margin Compression %'] > 0,
    (df_merged_prediction['Margin Compression %'] <= 0) &
    (df_merged_prediction['Margin Compression %'] > -20),
    (df_merged_prediction['Margin Compression %'] <= -20) &
    (df_merged_prediction['Margin Compression %'] > -50),
    df_merged_prediction['Margin Compression %'] <= -50
]

choices = [
    'Low Risk',
    'Moderate Risk',
    'High Risk',
    'Critical Risk'
]

df_merged_prediction['Risk Level'] = np.select(
    conditions,
    choices,
    default='Unknown'
)

In [13]:
# Merchant Becomes Unprofitable
df_merged_prediction['Unprofitable'] = np.where(
   df_merged_prediction['Future Margin'] < 0,
   'Yes',
   'No'
)


In [14]:
df_merged_prediction.head()

,Merchant ID,Company Name,MCC code,Business Industry,Price Model,MSF CC %,MSF DC $,State,PostCode,Default Brand,...,Tap TTV,EFTPOS Non-Tap TTV,Future Tap Revenue,Future EFTPOS Revenue,Future Revenue,Future Margin,Margin Compression $,Margin Compression %,Risk Level,Unprofitable
0,SP7030000005012,Anatolia Gozleme Kitchen,5812,"Eating Places, Restaurants",SmartCharge,1.20,0.30,VIC,3181.0,Smartpay,...,134290.282875,201.940275,940.031980,0.403881,940.435861,-202.374139,-516.334139,-164.458574,Critical Risk,Yes
1,SP7040000004162,Aussizz Migration And Education acc 2,8111,"Attorneys, Legal Services",SmartCharge,1.65,0.30,QLD,4000.0,Smartpay,...,6719.130000,0.000000,60.472170,0.000000,60.472170,58.092170,-74.477830,-56.180003,Critical Risk,No
2,SP7030000005066,Bella and Home Decor,5712,"Furniture, Home Furnishings, and Equipment Sto...",Simple rate,1.05,0.24,VIC,3803.0,Smartpay,...,22937.051900,930.848100,206.433467,2.792544,209.226011,113.916011,-330.843989,-74.387083,Critical Risk,No
3,SP7030000004158,Afghan Kebab Campbellfield,5812,"Eating Places, Restaurants",SmartCharge,1.10,0.25,VIC,3061.0,Smartpay,...,120790.007910,546.012090,845.530055,1.092024,846.622080,447.672080,-350.367920,-43.903554,High Risk,No
4,SP7030000002212,Byblos Market,5499,Misc. Food Stores - Convenience & Specialty Ma...,SmartCharge,1.65,0.30,VIC,3023.0,Smartpay,...,7335.990000,0.000000,66.023910,0.000000,66.023910,50.093910,-77.356090,-60.695245,Critical Risk,No


In [15]:
# =====================================================
# INDUSTRY SUMMARY
# =====================================================

df_Industry_prediction = (
    df_merged_prediction
    .groupby('Business Industry', as_index=False)
    .agg({
        'Value of Transactions': 'sum',
        'MSF Margin': 'sum',
        'Future Margin': 'sum',
        'Margin Compression $': 'sum'
    })
)

df_Industry_prediction['Margin Compression %'] = (
    df_Industry_prediction['Margin Compression $']
    / df_Industry_prediction['MSF Margin']
)

In [16]:
df_Industry_prediction.head()

,Business Industry,Value of Transactions,MSF Margin,Future Margin,Margin Compression $,Margin Compression %
0,"Accessories - Women's Accessory, Specialty Shops",1096094.25,8406.52,4783.037959,-3623.482041,-0.431032
1,Accessory Shops - (Not Elsewhere Classified),1544463.65,12085.06,6342.718405,-5742.341595,-0.475160
2,Accounting Services,144708.03,2413.88,818.146730,-1595.733270,-0.661066
3,Active Wear - Sports Apparel,529412.17,3425.82,2226.285882,-1199.534118,-0.350145
4,Agricultural Co-operatives,961178.06,4948.20,4062.719389,-885.480611,-0.178950


In [17]:
# =====================================================
# PORTFOLIO SUMMARY
# =====================================================

portfolio_summary = pd.DataFrame({
    'Current Margin': [df_merged_prediction['MSF Margin'].sum()],
    'Future Margin': [df_merged_prediction['Future Margin'].sum()],
    'Margin Compression $': [df_merged_prediction['Margin Compression $'].sum()]
})

portfolio_summary['Margin Compression %'] = (
    portfolio_summary['Margin Compression $']
    / portfolio_summary['Current Margin']
)


In [18]:
portfolio_summary.head()

,Current Margin,Future Margin,Margin Compression $,Margin Compression %
0,3979770.12,1.750118e+06,-2.229652e+06,-0.560246


In [19]:
df_merged_prediction.to_csv('df_merged_prediction_May.csv', index=False)
df_Industry_prediction.to_csv('df_Industry_prediction_May.csv', index=False)
portfolio_summary.to_csv('portfolio_summary_May.csv', index=False)

In [20]:
df_Industry_prediction.sort_values('Margin Compression %').head()


,Business Industry,Value of Transactions,MSF Margin,Future Margin,Margin Compression $,Margin Compression %
23,Bakeries,31181138.00,75553.29,-47803.405555,-123356.695555,-1.632711
182,"Stationery, Office Supplies, Printing, and Wri...",4569.29,105.05,-29.000000,-134.050000,-1.276059
201,Vocational Schools and Trade Schools,11463.95,306.48,-39.488200,-345.968200,-1.128844
195,Travel Agencies and Tour Operations,15182.26,175.92,-22.643718,-198.563718,-1.128716
48,"Civic, Fraternal, and Social Associations",287.00,36.56,-2.700000,-39.260000,-1.073851
